# Four Agents, No Ground Truth: An LLM Evaluation Problem

I built an adversarial multi-agent pipeline for scientific manuscript evaluation.
Four role-based subagents — biologist, bioinformatician, reviewer, editor —
generate competing interpretations and structured critiques.

The easy part: building it.  
The hard part: how do you measure whether iterative critique actually *improves* quality 
when there is no objective ground truth?

This is the central problem of AI evaluation for open-ended tasks. 
I am building toward a solution using DSPy and LangGraph.

---

## The Architecture

In [ ]:
# Pipeline schema — no API calls, just structure
pipeline = {
    "biologist": {
        "role": "Interprets results from biological first principles",
        "style": "Generative — proposes the most plausible biological interpretation",
        "output": "Interpretation memo + confidence scores per claim"
    },
    "bioinformatician": {
        "role": "Critiques analytical choices and statistical approach",
        "style": "Adversarial — challenges assumptions, flags confounds",
        "output": "Critique memo + severity ratings (1–5) per issue"
    },
    "reviewer": {
        "role": "Evaluates scientific novelty and rigour (LLM-as-Judge)",
        "style": "Structured — scores against explicit rubric",
        "output": "Rubric scores + justifications + priority improvements"
    },
    "editor": {
        "role": "Assesses clarity, structure, and narrative coherence (LLM-as-Judge)",
        "style": "Integrative — synthesises all prior critiques",
        "output": "Revision letter + ranked action items"
    }
}

print("=" * 58)
print(" ADVERSARIAL MANUSCRIPT EVALUATION PIPELINE")
print("=" * 58)
for agent, info in pipeline.items():
    print(f"\n[{agent.upper()}]")
    for k, v in info.items():
        print(f"  {k:12s}: {v}")
print("\n" + "=" * 58)
print(" Orchestration: LangGraph state machine")
print(" Optimisation:  DSPy (prompt pipeline tuning)")
print("=" * 58)

---

## An Example Exchange (Anonymised)

The following is a real excerpt from an evaluation run on a behavioural genomics manuscript, 
lightly anonymised. The manuscript reports a circuit manipulation that increases feeding 
behaviour and interprets this as evidence for a specific hunger-encoding circuit.

---

**[BIOLOGIST]** *Interpretation memo, Round 1:*

> The increase in feeding duration following optogenetic activation of DR5-HT neurons 
> is consistent with a serotonergic hunger gate. The result aligns with prior literature 
> showing DR serotonin suppression of satiety in *Drosophila* (Albin et al. 2015). 
> The authors' causal claim is plausible given the bidirectionality of the effect 
> (activation increases feeding; silencing decreases it).
> **Confidence: 0.72.** Main uncertainty: no proof that increased feeding reflects 
> genuine hunger state change versus isolated motor facilitation.

---

**[BIOINFORMATICIAN]** *Critique of Biologist memo, Round 1:*

> I flag a confound in the causal interpretation: the optogenetic stimulation protocol 
> (15s on, 45s off) may be driving feeding via motor entrainment rather than metabolic 
> state change. The lack of inter-bout interval analysis means we cannot distinguish 
> "hungrier" from "less able to stop eating." **Severity: 4/5.** 
> Required addition: bout microstructure analysis + comparison to 24h starved profile 
> across multiple metrics (not just total intake).

---

**[REVIEWER]** *Rubric scores after Round 1:*

| Dimension | Score (0–10) | Note |
|-----------|:------------:|------|
| Novelty | 7 | Circuit well-characterised; state claim is new |
| Rigour | 5 | Single-metric evidence for multi-metric claim |
| Clarity | 8 | Methods section exemplary |
| Impact | 7 | Framework has broad applicability |
| **Overall** | **6.75** | Major revision recommended |

---

The bioinformatician's critique (bout microstructure analysis) directly caused a 
revision to the manuscript — the authors added a DESTRA comparison in revision. 
The reviewer score improved from 6.75 to 7.9 in Round 2.

---

## What Makes This Hard

For a chess engine, ground truth is easy: did it win?
For manuscript quality: who decides what "better" means?

This is why I am building with **DSPy** — it allows optimising prompt pipelines against 
evaluation metrics. But the fundamental question remains: what metric captures 
*"scientifically rigorous and clearly communicated"*?

The answer I am working toward: bootstrap-estimated inter-rater agreement between 
human expert reviewers and the LLM judges, with bias correction 
(Rogan–Gladen estimator) for imperfect raters on both sides.

**Yes — this is DABEST applied to AI evaluation.**

The same framework for measuring "does this manipulation produce genuine hunger" 
also works for "does this agent produce genuine quality assessment." 
In both cases, the solution is ground-truth-referenced, multidimensional evaluation. 
In both cases, you need a reference distribution. In both cases, you need uncertainty quantification.

The measurement problem is the same problem.

---

## Status & Stack

| Component | Status |
|-----------|--------|
| 4-agent pipeline (Claude API) | ✓ Running |
| LangGraph state machine | ✓ Running |
| DSPy optimisation loop | In progress |
| Bootstrap inter-rater metric | In progress |
| Human expert calibration study | Planned |

Repository will be made public after the *Science Advances* submission.